# Playbook Soccer Analytics — GPU Test (Google Colab)

**Before running:** Go to `Runtime → Change runtime type` and select **T4 GPU**.

You will need:
- A **Roboflow API key** (free at roboflow.com) stored in Colab Secrets as `ROBOFLOW_API_KEY`
- A short soccer video clip (MP4, ideally 10–30 seconds for a quick test)

**Pipeline highlights (good-baseline-may9):**
- BoTSort tracker + appearance ReID (`yolo11n-cls.pt`)
- IDStabilizer — re-links IDs after occlusions using position + torso appearance
- Color-based team classification (fast, no GPU needed for this step)
- BallSmoother — interpolates missing ball detections
- HomographyStateMachine — holds last good homography through short failures
- KPI summary JSON/CSV alongside the annotated video

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
if gpu.returncode == 0:
    print('GPU detected:', gpu.stdout.strip())
else:
    print('⚠️  No GPU found.\n'
          'Go to Runtime → Change runtime type → T4 GPU, then re-run all cells.')

In [ ]:
# ── Cell 2: System packages ───────────────────────────────────────────────────
!apt-get install -qq ffmpeg libglib2.0-0 libsm6 libxext6 libxrender-dev

In [ ]:
# ── Cell 3: Python dependencies ───────────────────────────────────────────────
# Swap out Colab's opencv for headless (avoids display-backend conflicts)
!pip uninstall -qqy opencv-python opencv-python-headless 2>/dev/null

!pip install -q \
    'numpy>=2.0.0,<2.4.0' \
    opencv-python-headless==4.10.0.84 \
    onnxruntime==1.20.1 \
    tqdm \
    'requests>=2.32.3' \
    'pydantic>=2.11.7,<2.12.0' \
    pydantic-settings==2.4.0 \
    python-dotenv==1.0.1 \
    'supervision==0.27.0.post2' \
    'inference==1.2.2' \
    'ultralytics>=8.4.37,<8.5.0' \
    'lap>=0.5.13,<0.6'

!pip install -q 'transformers>=5.2.0,<5.3.0'

# Roboflow sports library (color-team helper, pitch config, annotators)
!pip install -q git+https://github.com/roboflow/sports.git@main

print('\n✅ All packages installed.')

In [ ]:
# ── Cell 4: Clone the repo ────────────────────────────────────────────────────
# Public repo — no token needed. If private, replace with:
#   !git clone https://<YOUR_TOKEN>@github.com/muwafagq/playbook-program.git /content/playbook
BRANCH = 'claude/setup-gpu-video-testing-JhgUH'
!git clone --branch {BRANCH} https://github.com/muwafagq/playbook-program.git /content/playbook

import os, sys
os.chdir('/content/playbook')
sys.path.insert(0, '/content/playbook')
active_branch = !git rev-parse --abbrev-ref HEAD
print('Working dir:', os.getcwd())
print('Branch:', active_branch[0])

In [ ]:
# ── Cell 5: API key + .env ────────────────────────────────────────────────────
# baseline.env is already in the repo with all tuned parameters including
# PLAYER_MODEL_ID and FIELD_MODEL_ID. We copy it to .env and explicitly export
# every key so nothing can be shadowed by a stale env var or config default.
import shutil, os
shutil.copy('baseline.env', '.env')

try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
    print('✅ Loaded API key from Colab Secrets')
except Exception:
    ROBOFLOW_API_KEY = 'YOUR_ROBOFLOW_API_KEY_HERE'   # fallback
    print('⚠️  Using hardcoded API key — prefer Colab Secrets')

# Patch the API key into .env
with open('.env', 'r') as f:
    env_text = f.read()
env_text = env_text.replace('ROBOFLOW_API_KEY=', f'ROBOFLOW_API_KEY={ROBOFLOW_API_KEY}')
with open('.env', 'w') as f:
    f.write(env_text)

# Explicitly export every key from baseline.env into os.environ so nothing
# can be shadowed by stale session variables or config.py defaults.
# ROBOFLOW_API_KEY and DEVICE are also set here for completeness.
from dotenv import dotenv_values
env_vals = dotenv_values('.env')
for k, v in env_vals.items():
    if v is not None:
        os.environ[k] = v

# Always force CUDA and the correct API key regardless of .env content.
os.environ['ROBOFLOW_API_KEY'] = ROBOFLOW_API_KEY
os.environ['DEVICE'] = 'cuda'

print('\n── Active model config ─────────────────────────────────')
print('PLAYER_MODEL_ID :', os.environ.get('PLAYER_MODEL_ID', '(not set — will use config.py default)'))
print('FIELD_MODEL_ID  :', os.environ.get('FIELD_MODEL_ID',  '(not set — will use config.py default)'))
print('TRACKER_TYPE    :', os.environ.get('TRACKER_TYPE', '(not set)'))
print('DEVICE          :', os.environ.get('DEVICE'))
print('────────────────────────────────────────────────────────')

In [ ]:
# ── Cell 6: Provide a test video ──────────────────────────────────────────────
# Choose ONE option and comment out the others.

# --- Option A: Upload a local file -------------------------------------------
from google.colab import files as colab_files
print('Select your MP4 file in the dialog below...')
uploaded = colab_files.upload()
VIDEO_PATH = '/content/' + list(uploaded.keys())[0]
print('Video ready at:', VIDEO_PATH)

# --- Option B: Download a YouTube clip (yt-dlp) ------------------------------
# !pip install -q yt-dlp
# YT_URL = 'https://www.youtube.com/watch?v=REPLACE_ME'
# !yt-dlp -o /content/test_clip.%(ext)s --recode-video mp4 -q "$YT_URL"
# VIDEO_PATH = '/content/test_clip.mp4'

# --- Option C: Mount Google Drive --------------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')
# VIDEO_PATH = '/content/drive/MyDrive/YOUR_FOLDER/your_clip.mp4'

In [ ]:
# ── Cell 7 (optional): Trim to first N seconds ────────────────────────────────
# Skip if your clip is already short (< 30 s).
TRIM_SECONDS = 20
TRIMMED_PATH = '/content/test_trimmed.mp4'
!ffmpeg -y -i "{VIDEO_PATH}" -t {TRIM_SECONDS} -c copy "{TRIMMED_PATH}" -loglevel warning
VIDEO_PATH = TRIMMED_PATH
print(f'Trimmed to {TRIM_SECONDS}s → {VIDEO_PATH}')

In [ ]:
# ── Cell 8: Run the pipeline ──────────────────────────────────────────────────
# All tuned parameters come from baseline.env / .env.
# TEAM_MODE=color is the default — fast, no heavy embedding model needed.
# Set enable_team=False to skip team classification entirely for a quicker sanity check.

OUT_DIR = '/content/outputs'

from main import main
main(
    source_video=VIDEO_PATH,
    out_dir=OUT_DIR,
    enable_team=True,
)

In [ ]:
# ── Cell 9: Preview annotated video ───────────────────────────────────────────
from IPython.display import HTML
from base64 import b64encode
import glob, os

# run.sh writes to outputs_test/<run_id>/; main() used OUT_DIR directly
video_file = OUT_DIR + '/annotated.mp4'
video_bytes = open(video_file, 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(video_bytes).decode()
HTML(f'<video width="800" controls><source src="{data_url}" type="video/mp4"></video>')

In [ ]:
# ── Cell 10: KPI summary ──────────────────────────────────────────────────────
import json, pandas as pd

kpi_json = OUT_DIR + '/kpi_summary.json'
if os.path.exists(kpi_json):
    with open(kpi_json) as f:
        kpi = json.load(f)
    print(json.dumps(kpi, indent=2))
else:
    print('kpi_summary.json not found — check OUT_DIR path')

csv_path = OUT_DIR + '/per_frame_tracks.csv'
df = pd.read_csv(csv_path)
print(f'\nTracking CSV: {len(df):,} rows | {df.frame.nunique()} frames | {df.track_id.nunique()} unique IDs')
df.head(5)

In [ ]:
# ── Cell 11: Download all outputs ─────────────────────────────────────────────
from google.colab import files as colab_files
for fname in ['annotated.mp4', 'per_frame_tracks.csv', 'kpi_summary.json', 'kpi_summary.csv']:
    fpath = f'{OUT_DIR}/{fname}'
    if os.path.exists(fpath):
        colab_files.download(fpath)
    else:
        print(f'Skipped (not found): {fpath}')